In [1]:
# Get raw data
with open('input/22.txt', 'r') as f:
    rawinput = f.read()

In [2]:
import numpy as np

In [3]:
# Part 1
def do_turn(pos, orient, turn):
    if turn in 'LR':
        pos = ([pos[1], board_dims[orient][0]-pos[0]-1] if turn=='L' 
               else [board_dims[orient][1]-pos[1]-1, pos[0]])
        orient = (orient + {'L':-1,'R':1}[turn]) % 4
    return pos, orient

board = [np.array([list((k+' '*j)[:j])
                   for i in [rawinput.split('\n\n')[0].split('\n')]
                   for j in [max([len(j) for j in i])]
                   for k in i])]
board_ix = [np.arange(board[-1].size).reshape(board[-1].shape)]
board_dims = [board[-1].shape]
for i in 'XXX':
    board += [np.flip(board[-1].T, axis=0)]
    board_ix += [np.flip(board_ix[-1].T, axis=0)]
    board_dims += [board[-1].shape]

directions = [[int((m:=i[k:l])[:-1]), m[-1]]
              for i in [rawinput.split('\n\n')[1].strip()+'X']
              for j in [[t
                         for t,(u,v) in enumerate(zip([*f'{i}0'],[*f'X{i}']))
                         if u.isnumeric() and not v.isnumeric()]]
              for k,l in zip(j,j[1:])]
pos = [0, np.argmax(board[0][0]=='.').item()]
orient = 0

for nsteps, turn in directions:
    px = np.mod(np.arange(xmax:=board_dims[orient][1]) + pos[1], xmax)
    psc = np.cumsum(np.vectorize({' ':0,'.':1,'#':99}.get)(board[orient][pos[0]][px]))-1
    pos[1] = px[psc == np.max(np.where(psc <= nsteps, psc, -1))][0].item()
    
    pos, orient = do_turn(pos, orient, turn)
    
sum([1000 * (board_ix[orient][*pos] // board_dims[orient][1] + 1),
     4 * (board_ix[orient][*pos] % board_dims[orient][1] + 1),
     orient]).item()

131052

In [4]:
# Part 2
edgecells = {(i,j,n): (k,l,n) 
             for n,(b,x,(h,w)) in enumerate([*zip(board,board_ix,board_dims)])
             for f in [((b != ' ') & (~np.pad(b[:, 1:]!=' ',((0,0),(0,1))))).reshape(-1)]
             for t,u,v in [[x.reshape(-1), board_dims[0][1], board_ix[0].reshape(-1)]]
             for i,j,k,l in np.column_stack((t//u, t%u, v//w, v%w))[f].tolist()}

eclist = {*edgecells}
eclist_ord = []
corners_inner = set()
corners_outer = set()
while eclist:
    rnklist = sorted([[sum(j[:2]), j[2], i] 
                      for i in eclist                       
                      for j in [[abs(k-l) 
                                 for k,l in zip(i,
                                                eclist_ord[-1] if eclist_ord else [*edgecells][0])]]])
    if rnklist[0][0]==2:
        corners_inner.update({*[eclist_ord[-1][:2], rnklist[0][2][:2]]})
    elif rnklist[0][0]==0:
        corners_outer.add(rnklist[0][2][:2])
    eclist_ord += rnklist[0][2:]
    eclist.discard(eclist_ord[-1])
    
edge_mv = {edgecells[q[0]]: tuple([j-k-1 
                                   for i in [edgecells[q[1]]]
                                   for j,k in zip(board_dims[i[2]], i[:2])]
                                  + [(2+edgecells[q[1]][2])%4])
           for s in [i+1
                     for i,j in enumerate(eclist_ord[1:]) 
                     if all([k in corners_inner 
                             for k in [j[:2],eclist_ord[i][:2]]])]
           for z in [eclist_ord[s:]+eclist_ord[:s]]
           for e in [i+1 
                     for i,j in enumerate(zip(z,z[::-1])) 
                     if all([k[:2] in corners_outer for k in j])][:1]
           for p in [*zip(z,z[::-1])][:e]
           for q in [p,p[::-1]]}

pos = [0, np.argmax(board[0]=='.').item()]
orient = 0
for dirn, (nsteps, turn) in enumerate(directions):
    nsleft = nsteps
    while nsleft:
        px = np.mod(np.arange(w:=board_dims[orient][1]) + pos[1], w)
        if ((y:=min(np.argmax(np.append(board[orient][pos[0]][px],'#')=='#').item()-1, nsleft))
            < (z:=np.argmax(board[orient][pos[0]][px]==' ').item())):
            pos[1] += y
            break
        else:
            pos[1] += z-1
            nsleft -= z-1
            newstate = edge_mv[tuple(pos+[orient])]
            if board[newstate[2]][*newstate[:2]]=='#':
                break
            else:
                *pos, orient = [*newstate]
                nsleft -= 1
    pos, orient = do_turn(pos, orient, turn)
    
sum([1000 * (board_ix[orient][*pos] // board_dims[orient][1] + 1),
     4 * (board_ix[orient][*pos] % board_dims[orient][1] + 1),
     orient]).item()

4578